## SSH and Diffleop Runtime Checks

Run the following cells inside the RunPod Web Terminal or over SSH after the Pod is running. The commands assume the repository is available at `/workspace/gpcr-diffleop` and Diffleop is the submodule at `/workspace/gpcr-diffleop/Diffleop`.


In [ ]:
apt-get update
apt-get install -y openssh-server

mkdir -p /run/sshd /var/run/sshd /root/.ssh
chmod 700 /root/.ssh
echo "$PUBLIC_KEY" > /root/.ssh/authorized_keys
chmod 600 /root/.ssh/authorized_keys

sed -i 's/^#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
sed -i 's/^#PubkeyAuthentication.*/PubkeyAuthentication yes/' /etc/ssh/sshd_config
/usr/sbin/sshd

ps aux | grep sshd


## Inspect the Current State

These commands check the mounted workspace, repository/submodule state, demo data, conda environment, and CUDA availability.


In [ ]:
hostname
date
df -h /workspace
ls -la /workspace

cd /workspace/gpcr-diffleop
git status --short
git submodule status --recursive || true

cd /workspace/gpcr-diffleop/Diffleop
ls -la
find ckpt -maxdepth 2 -type f -printf "%p %s bytes\n" 2>/dev/null || true
find data/demo -maxdepth 2 -type f | sort | head -50

/opt/conda/bin/mamba env list


In [ ]:
cd /workspace/gpcr-diffleop/Diffleop

/opt/conda/bin/mamba run -n diffleop python - <<'PY'
import sys
print("python", sys.version)

mods = ["torch", "torch_geometric", "rdkit", "lmdb", "numpy", "scipy", "yaml", "alphaspace2", "mdtraj", "torchdrug"]
for name in mods:
    try:
        mod = __import__(name)
        print(name, "OK", getattr(mod, "__version__", ""))
    except Exception as exc:
        print(name, "FAIL", type(exc).__name__, exc)

import torch
print("cuda_available", torch.cuda.is_available())
print("cuda_version", torch.version.cuda)
print("cuda_device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
PY


In [ ]:
cd /workspace/gpcr-diffleop/Diffleop

/opt/conda/bin/mamba run -n diffleop python -c 'from models.diffleop import Diffleop; from datasets.pl_pair_dataset_affinity import get_dataset_dec_aff; import utils.transforms as trans; print("diffleop imports ok")'


## Download Checkpoint and Run a Minimal Sampling Test

The sampling config requires `ckpt/diffleop_dec.pt`. The checkpoint is downloaded from the Zenodo record referenced by the Diffleop README. This smoke test uses one demo item and one sample. Note that `scripts/sample.py` still runs the model checkpoint diffusion schedule, so it may use 1000 internal sampling steps even if the temporary config sets `num_steps` lower.


In [ ]:
cd /workspace/gpcr-diffleop/Diffleop
mkdir -p ckpt

python3 - <<'PY'
import urllib.request

url = "https://zenodo.org/api/records/14210941/files/diffleop_dec.pt/content"
out = "ckpt/diffleop_dec.pt"
print("downloading", url)
urllib.request.urlretrieve(url, out)
print("saved", out)
PY

ls -lh ckpt/diffleop_dec.pt


In [ ]:
cd /workspace/gpcr-diffleop/Diffleop

/opt/conda/bin/mamba run -n diffleop python - <<'PY'
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path("configs/sampling_dec.yml").read_text())
cfg["sample"]["num_samples"] = 1
cfg["sample"]["num_steps"] = 20
Path("/tmp/sampling_dec_smoke.yml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path("/tmp/sampling_dec_smoke.yml").read_text())
PY

/opt/conda/bin/mamba run -n diffleop python -W ignore scripts/sample.py \
  /tmp/sampling_dec_smoke.yml \
  -i 1 \
  --device cuda:0 \
  --type dec

find outputs -maxdepth 4 -type f | sort | head -50
